In [79]:
import polars as pl
import os
import glob

## Functions

In [80]:
def reorder_cols(df: pl.DataFrame) -> pl.DataFrame:
    cols = df.columns

    if "sample" in cols:
        cols.remove("sample")
        cols.insert(0, "sample")

    return df[cols]

In [81]:
def summarize_res(df: pl.DataFrame) -> pl.DataFrame:
    
    bool_cols = df['filter_1_mutation_intra_hairpin_loop':'filter_8_low_quality'].columns
    str_cols = df['msec_filter_123':'msec_filter_all'].columns
    n_variants = df.height

    # 1. Initialize a dictionary with empty lists for your columns
    results = {
        "filter": [],
        "percentage": [],
        "n_failed": []
    }

    # 2. Populate the dictionary inside your loops
    for col in bool_cols:
        n_failed = df[col].sum() # Sum counts True values
        
        results["filter"].append(f"%_{col}")
        results["percentage"].append((n_failed / n_variants) * 100)
        results["n_failed"].append(n_failed)

    for col in str_cols:
        n_failed = df.filter(~pl.col(col).is_null()).height # Count non-nulls
        
        results["filter"].append(f"%_{col}")
        results["percentage"].append((n_failed / n_variants) * 100)
        results["n_failed"].append(n_failed)

    # 3. Create DataFrame directly from the dictionary
    return pl.DataFrame(results).with_columns(pl.lit(n_variants).alias("total_varints"))

## Main

In [82]:
msec_paths = sorted(glob.glob("../vcf-micr-svf/*/*.microsec.tsv"))

all_res = []

for i, path in enumerate(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x : x.lower())
    
	all_res.append(df)
	print(f"{i+1} Processed {sample}")
    
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

1 Processed ORD-1992722-01
2 Processed ORD-1992968-01
3 Processed ORD-1993016-01
4 Processed ORD-1993732-01
5 Processed ORD-1993783-01
6 Processed ORD-1994198-01
7 Processed ORD-1994251-01
8 Processed ORD-1994255-01
9 Processed ORD-1994335-01
10 Processed ORD-1994600-01
11 Processed ORD-1994629-01
12 Processed ORD-1994630-01
13 Processed ORD-1994651-01
14 Processed ORD-1994723-01
15 Processed ORD-1995002-01
16 Processed ORD-1995012-01
17 Processed ORD-1995110-01
18 Processed ORD-1995114-01
19 Processed ORD-1996633-01
20 Processed ORD-1996744-01
21 Processed ORD-1996747-01
22 Processed ORD-1997438-01
23 Processed ORD-1997463-01
24 Processed ORD-1998350-01
25 Processed ORD-1998420-01
26 Processed ORD-1998424-01
27 Processed ORD-1998426-01
28 Processed ORD-1998431-01
29 Processed ORD-1998452-01
30 Processed ORD-1999058-01
31 Processed ORD-1999155-01
32 Processed ORD-1999159-01
33 Processed ORD-1999175-01
34 Processed ORD-1999177-01
35 Processed ORD-1999178-01
36 Processed ORD-1999183-01
3

In [83]:
# All artifacts
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1992968-01""","""1-snv""","""chr7""",92462584,"""C""","""A""","""N""","""CCATAGGCGCCCTCCCCGATATCCGCCACG…",144,159,88,0,143,122,71,265,263,0.081019,0.127673,0.054717,0.0,0.553459,1.0,0.001394,0.000002,false,false,false,false,true,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1993732-01""","""1-snv""","""chr6""",106553602,"""G""","""A""","""N""","""CTCCCACGGCGGGAACAGCCACCACGGCAG…",144,590,311,0,143,141,71,351,293,0.019444,0.021186,0.021525,0.0,0.527119,1.0,1.0,0.142659,false,false,false,false,true,false,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1993783-01""","""1-snv""","""chr2""",209116217,"""C""","""T""","""N""","""TAATCAATTCCCAAATGATTTGTGTCATTT…",101,1342,55,0,100,100,50,347,359,0.019131,0.019374,0.017511,0.489568,0.040984,1.0,1.0,1.0,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1994335-01""","""1-snv""","""chr9""",98278972,"""T""","""C""","""Y""","""CTCCGTTTTCTTCTTCTTCTCCTCCTCCTC…",144,5763,1682,0,143,143,71,350,392,0.04707,0.070154,0.020718,0.0,0.291862,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1994600-01""","""1-snv""","""chr8""",117862875,"""T""","""A""","""Y""","""ACCTCTTCCTCTTCATCATCATCTTTTTCC…",144,3998,385,0,143,143,71,366,358,0.033523,0.037919,0.021111,0.0,0.096298,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""1-snv""","""chr17""",29483000,"""G""","""T""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-2131606-01""","""2-ins""","""chr8""",117864264,"""C""","""CAG""","""N""","""TGGTGGAGGCATAGCTGACTCAGATCTATG…",144,37,26,0,118,112,56,145,112,0.010323,0.005405,0.010811,0.0,0.702703,9.2195e-9,0.000591,0.001694,true,false,false,false,true,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null


110

In [84]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,mut_type,chr,pos,ref,alt,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,str,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1993783-01""","""1-snv""","""chr2""",209116217,"""C""","""T""","""N""","""TAATCAATTCCCAAATGATTTGTGTCATTT…",101,1342,55,0,100,100,50,347,359,0.019131,0.019374,0.017511,0.489568,0.040984,1.0,1.0,1.0,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1994651-01""","""1-snv""","""chr2""",25462068,"""A""","""G""","""N""","""CTGACACTTCTTTGGCATCAGTCATCACAG…",144,43,0,0,94,141,68,94,144,0.040859,0.037209,0.037209,0.0,0.0,0.101987,2.5514e-10,1.4567e-10,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1998350-01""","""1-snv""","""chr3""",185191097,"""G""","""A""","""N""","""ATCCCAGACTCAATATGCACAGACAGGACA…",144,37,0,0,102,129,69,403,129,0.010135,0.013514,0.0,0.0,0.0,0.00014,6.0713e-7,4.5437e-7,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1998350-01""","""1-snv""","""chrX""",39914756,"""C""","""T""","""N""","""GTGATCGTTCTCAACAGCATTGTGCAGAGG…",144,52,0,0,141,128,46,161,305,0.007212,0.013462,0.003846,0.0,0.0,1.9184e-10,0.05919,0.080462,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1998420-01""","""18-ins""","""chr5""",79950727,"""G""","""GCAGCGCCCGCAGCGCCCC""","""Y""","""CTGCAGCGGCCGCAGCGGCCGCAGCGCCCG…",144,93,17,0,89,92,63,225,204,0.046819,0.013978,0.068817,0.0,0.182796,8.8682e-28,4.5603e-10,3.7602e-43,true,false,true,false,false,true,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""1-snv""","""chr3""",178952085,"""A""","""G""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""1-snv""","""chr2""",25463568,"""A""","""G""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,71,156,117,0.008433,0.010714,0.0125,0.0,0.035714,3.0944e-7,4.7255e-11,9.2555e-11,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 1: p is small, but sup…"
"""ORD-2131606-01""","""16-del""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"


41

In [85]:
summarize_res(final_df)

filter,percentage,n_failed,total_varints
str,f64,i64,i32
"""%_filter_1_mutation_intra_hair…",1.068091,32,2996
"""%_filter_2_hairpin_structure""",0.0,0,2996
"""%_filter_3_microhomology_induc…",0.834446,25,2996
"""%_filter_4_highly_homologous_r…",0.200267,6,2996
"""%_filter_5_soft_clipped_reads""",0.901202,27,2996
…,…,…,…
"""%_filter_7_mutation_at_homopol…",0.033378,1,2996
"""%_filter_8_low_quality""",2.269693,68,2996
"""%_msec_filter_123""",1.502003,45,2996


### XML

In [86]:
msec_paths = sorted(glob.glob("../xml-micr-svf/*/*.microsec.tsv"))
len(msec_paths)

210

In [87]:
all_res = []

for i, path in enumerate(msec_paths):
	sample = os.path.basename(path).replace(".microsec.tsv", "")
	df = pl.read_csv(path, separator="\t").rename(lambda x: x.lower())
    
	all_res.append(df)
	print(f"{i+1} Processed {sample}")
    
final_df = pl.concat(all_res, how="vertical_relaxed").pipe(reorder_cols)

1 Processed ORD-1992722-01
2 Processed ORD-1992968-01
3 Processed ORD-1993016-01
4 Processed ORD-1993732-01
5 Processed ORD-1993783-01
6 Processed ORD-1994198-01
7 Processed ORD-1994251-01
8 Processed ORD-1994255-01
9 Processed ORD-1994335-01
10 Processed ORD-1994600-01
11 Processed ORD-1994629-01
12 Processed ORD-1994630-01
13 Processed ORD-1994651-01
14 Processed ORD-1994723-01
15 Processed ORD-1995002-01
16 Processed ORD-1995012-01
17 Processed ORD-1995110-01
18 Processed ORD-1995114-01
19 Processed ORD-1996633-01
20 Processed ORD-1996744-01
21 Processed ORD-1996747-01
22 Processed ORD-1997438-01
23 Processed ORD-1997463-01
24 Processed ORD-1998350-01
25 Processed ORD-1998420-01
26 Processed ORD-1998424-01
27 Processed ORD-1998426-01
28 Processed ORD-1998431-01
29 Processed ORD-1998452-01
30 Processed ORD-1999058-01
31 Processed ORD-1999155-01
32 Processed ORD-1999159-01
33 Processed ORD-1999175-01
34 Processed ORD-1999177-01
35 Processed ORD-1999178-01
36 Processed ORD-1999183-01
3

In [88]:
final_df

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1992722-01""","""chr1""",120478170,"""G""","""A""","""NOTCH2""",false,1373,"""3580C>T""","""Q1194*""",0.0036,"""nonsense""","""NM_024408""","""-""",false,"""1-snv""","""N""","""GCCTCCATTCTGGCAGGGCTAATTCTGGCA…",144,43,2,0,120,123,68,120,298,0.016634,0.023256,0.018605,0.0,0.046512,0.254119,0.006638,0.000024,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1992722-01""","""chr1""",162724439,"""G""","""T""","""DDR2""",true,3723,"""211G>T""","""A71S""",0.0016,"""missense""","""NM_006182""","""+""",false,"""1-snv""","""N""","""ACTCAGAAGAAGGGGATGGATCCTGGTGCC…",144,99,7,0,143,143,70,160,329,0.078774,0.074747,0.074747,0.0,0.070707,0.447462,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1992722-01""","""chr10""",104854199,"""T""","""C""","""NT5C2""",true,1062,"""827A>G""","""H276R""",0.0235,"""missense""","""NM_001134373""","""-""",false,"""1-snv""","""N""","""AGTAGGACTGCCATGGTCGACGGGAGCTCC…",144,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1992722-01""","""chr11""",69625302,"""C""","""A""","""FGF3""",true,1657,"""491G>T""","""R164L""",0.0036,"""missense""","""NM_005247""","""-""",false,"""1-snv""","""N""","""GGCGGGTCTTGAAGCCCCTGAGGGGCCGGC…",144,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-1992722-01""","""chr16""",341269,"""G""","""A""","""AXIN1""",true,765,"""2215C>T""","""R739C""",0.5412,"""missense""","""NM_003502""","""-""",false,"""1-snv""","""N""","""CGCTGGCCTGACGCAGGCGCATCCCCGCCG…",144,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2133291-01""","""chr17""",37856504,"""G""","""A""","""ERBB2""",true,3682,"""13G>A""","""A5T""",0.0019,"""missense""","""NM_004448""","""+""",false,"""1-snv""","""N""","""TGAGCACCATGGAGCTGGCGACCTTGTGCC…",144,195,1,0,133,139,69,179,414,0.039672,0.048718,0.010256,0.0,0.005128,1.2908e-7,1.7741e-8,1.2642e-8,false,false,false,false,false,false,false,false,null,null,null,""" filter 1: p is small, but sup…"
"""ORD-2133291-01""","""chr22""",29108003,"""C""","""T""","""CHEK2""",true,3729,"""686G>A""","""G229D""",0.0013,"""missense""","""NM_007194""","""-""",false,"""1-snv""","""N""","""GCTTTACCTCTCCACAGGCATCACTAGAGG…",144,36,0,0,140,140,52,140,338,0.009452,0.016667,0.005556,0.0,0.0,0.000018,0.405816,0.401138,false,false,false,false,false,false,false,false,null,null,null,null
"""ORD-2133291-01""","""chr4""",1920021,"""A""","""C""","""WHSC1""",true,1979,"""1081A>C""","""K361Q""",0.5073,"""missense""","""NM_133335""","""+""",false,"""1-snv""","""N""","""ATCTCAACCCTCAAGTAGCCCAGGAGGCTG…",144,7806,748,0,143,143,71,703,734,0.013053,0.014758,0.015168,0.0,0.095824,1.0,1.0,1.0,false,false,false,false,false,false,false,false,null,null,null,null


In [89]:
# All artifacts
arti_all_filter = final_df.filter(~pl.col("msec_filter_all").is_null())
display(arti_all_filter)
# Number of Samples with >= 1 artifacts
arti_all_filter["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1992968-01""","""chr7""",92462584,"""C""","""A""","""CDK6""",true,4949,"""54G>T""","""E18D""",0.0024,"""missense""","""NM_001259""","""-""",false,"""1-snv""","""N""","""CCATAGGCGCCCTCCCCGATATCCGCCACG…",144,159,88,0,143,122,71,265,263,0.081019,0.127673,0.054717,0.0,0.553459,1.0,0.001394,0.000002,false,false,false,false,true,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-1993732-01""","""chr6""",106553602,"""G""","""A""","""PRDM1""",false,1143,"""1567G>A""","""A523T""",0.0577,"""missense""","""NM_001198""","""+""",false,"""1-snv""","""N""","""CTCCCACGGCGGGAACAGCCACCACGGCAG…",144,590,311,0,143,141,71,351,293,0.019444,0.021186,0.021525,0.0,0.527119,1.0,1.0,0.142659,false,false,false,false,true,false,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1993783-01""","""chr2""",209116217,"""C""","""T""","""IDH1""",true,1098,"""59G>A""","""R20Q""",0.4199,"""missense""","""NM_005896""","""-""",false,"""1-snv""","""N""","""TAATCAATTCCCAAATGATTTGTGTCATTT…",101,1342,55,0,100,100,50,347,359,0.019131,0.019374,0.017511,0.489568,0.040984,1.0,1.0,1.0,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1994335-01""","""chr9""",98278972,"""T""","""C""","""PTCH1""",false,845,"""131A>G""","""E44G""",0.6793,"""missense""","""NM_001083603""","""-""",false,"""1-snv""","""Y""","""CTCCGTTTTCTTCTTCTTCTCCTCCTCCTC…",144,5763,1682,0,143,143,71,350,392,0.04707,0.070154,0.020718,0.0,0.291862,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
"""ORD-1994600-01""","""chr8""",117862875,"""T""","""A""","""RAD21""",true,843,"""1602A>T""","""E534D""",0.516,"""missense""","""NM_006265""","""-""",false,"""1-snv""","""Y""","""ACCTCTTCCTCTTCATCATCATCTTTTTCC…",144,3998,385,0,143,143,71,366,358,0.033523,0.037919,0.021111,0.0,0.096298,1.0,1.0,1.0,false,false,false,false,false,true,false,false,null,null,"""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2131606-01""","""chr17""",29483000,"""G""","""T""","""NF1""",false,2006,"""61-1G>T""","""splice site 61-1G>T""",0.0957,"""splice""","""NM_001042492""","""+""",false,"""1-snv""","""N""","""TTTTTTTTCTTTTTTTTTCATCTTCCAATA…",144,2299,464,0,143,143,71,599,596,0.070046,0.115789,0.038669,0.0,0.201827,1.0,1.0,1.0,false,false,false,false,false,false,false,true,null,null,"""Artifact suspicious""",null
"""ORD-2131606-01""","""chr2""",225422558,"""TTTCATCCATGGTCATC""","""T""","""CUL3""",false,1027,"""67-1_81delGATGACCATGGATGAA""","""splice site 67-1_81delGATGACCA…",0.0351,"""splice""","""NM_003590""","""-""",false,"""16-del""","""N""","""CCAAATGCTGTTTACATATTTTGTAATATC…",144,185,52,0,125,126,80,277,141,0.069294,0.115676,0.046486,0.0,0.281081,2.0150e-34,6.6014e-10,0.000001,true,false,false,false,false,false,false,true,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",""" filter 3: p is small, but sup…"
"""ORD-2131606-01""","""chr8"

102

In [90]:
# Filter 1234
arti_filter_1234 = final_df.filter(~pl.col("msec_filter_1234").is_null())
display(arti_filter_1234)
# Number of Samples with >= 1 artifacts
arti_filter_1234["sample"].n_unique()

sample,chrom,pos,ref,alt,gene,is_vus,depth,cds_effect,protein_effect,allele_fraction,functional_effect,transcript,strand,equivocal,mut_type,simplerepeat_trf,neighborhood_sequence,read_length,total_read,soft_clipped_read,flag_hairpin,pre_support_length,post_support_length,short_support_length,pre_farthest,post_farthest,low_quality_base_rate_under_q18,low_quality_pre,low_quality_post,distant_homology_rate,soft_clipped_rate,prob_filter_1,prob_filter_3_pre,prob_filter_3_post,filter_1_mutation_intra_hairpin_loop,filter_2_hairpin_structure,filter_3_microhomology_induced_mutation,filter_4_highly_homologous_region,filter_5_soft_clipped_reads,filter_6_simple_repeat,filter_7_mutation_at_homopolymer,filter_8_low_quality,msec_filter_123,msec_filter_1234,msec_filter_all,comment
str,str,i64,str,str,str,bool,i64,str,str,f64,str,str,str,bool,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,bool,bool,bool,bool,bool,bool,bool,bool,str,str,str,str
"""ORD-1993783-01""","""chr2""",209116217,"""C""","""T""","""IDH1""",true,1098,"""59G>A""","""R20Q""",0.4199,"""missense""","""NM_005896""","""-""",false,"""1-snv""","""N""","""TAATCAATTCCCAAATGATTTGTGTCATTT…",101,1342,55,0,100,100,50,347,359,0.019131,0.019374,0.017511,0.489568,0.040984,1.0,1.0,1.0,false,false,false,true,false,false,false,false,null,"""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1994651-01""","""chr2""",25462068,"""A""","""G""","""DNMT3A""",true,1475,"""2339T>C""","""I780T""",0.0027,"""missense""","""NM_022552""","""-""",false,"""1-snv""","""N""","""CTGACACTTCTTTGGCATCAGTCATCACAG…",144,43,0,0,94,141,68,94,144,0.040859,0.037209,0.037209,0.0,0.0,0.101987,2.5514e-10,1.4567e-10,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1998350-01""","""chr3""",185191097,"""G""","""A""","""MAP3K13""",true,1972,"""1978G>A""","""G660R""",0.003,"""missense""","""NM_004721""","""+""",false,"""1-snv""","""N""","""ATCCCAGACTCAATATGCACAGACAGGACA…",144,37,0,0,102,129,69,403,129,0.010135,0.013514,0.0,0.0,0.0,0.00014,6.0713e-7,4.5437e-7,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1998350-01""","""chrX""",39914756,"""C""","""T""","""BCOR""",true,525,"""4504G>A""","""D1502N""",0.0095,"""missense""","""NM_017745""","""-""",false,"""1-snv""","""N""","""GTGATCGTTCTCAACAGCATTGTGCAGAGG…",144,52,0,0,141,128,46,161,305,0.007212,0.013462,0.003846,0.0,0.0,1.9184e-10,0.05919,0.080462,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-1998426-01""","""chrX""",44833932,"""AC""","""A""","""KDM6A""",false,924,"""357delC""","""Y120fs*60""",0.0043,"""frameshift""","""NM_021140""","""+""",false,"""1-del""","""N""","""ATTATCTGCATACCAGAGGTATACAGTTTA…",144,35,4,0,134,134,50,241,148,0.006349,0.011429,0.002857,0.0,0.114286,9.2924e-11,0.033455,0.065364,true,false,false,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ORD-2117808-01""","""chr3""",178952085,"""A""","""G""","""PIK3CA""",false,703,"""3140A>G""","""H1047R""",0.01,"""missense""","""NM_006218""","""+""",false,"""1-snv""","""N""","""GAAACAAATGAATGATGCACGTCATGGTGG…",101,29,0,0,60,90,49,219,326,0.017071,0.013793,0.027586,0.0,0.0,0.001003,3.1723e-9,3.0091e-9,false,false,true,false,false,false,false,false,"""Artifact suspicious""","""Artifact suspicious""","""Artifact suspicious""",null
"""ORD-2118842-01""","""chr2""",25463568,"""A""","""G""","""DNMT3A""",true,2024,"""2114T>C""","""I705T""",0.004,"""missense""","""NM_022552""","""-""",false,"""1-snv""","""N""","""CATTGCAGGGACTGCCCCCAGTCACCAGAT…",144,56,2,0,95,117,71,156,117,0.008433,0.010714,0.0125,0.0,0.035714,3.0944e-7,4.7255e-11,9.2555e-11,false,false,true,fal

36

In [91]:
summarize_res(final_df)

filter,percentage,n_failed,total_varints
str,f64,i64,i32
"""%_filter_1_mutation_intra_hair…",0.981105,27,2752
"""%_filter_2_hairpin_structure""",0.0,0,2752
"""%_filter_3_microhomology_induc…",0.763081,21,2752
"""%_filter_4_highly_homologous_r…",0.25436,7,2752
"""%_filter_5_soft_clipped_reads""",0.90843,25,2752
…,…,…,…
"""%_filter_7_mutation_at_homopol…",0.036337,1,2752
"""%_filter_8_low_quality""",2.325581,64,2752
"""%_msec_filter_123""",1.453488,40,2752
